# Think@n Analysis (Appendix C, Figure 8)

Reproduces **Figure 8** from Appendix C of the DTR paper.

This notebook performs a detailed analysis of the Think@n strategy by varying:
- **n**: number of candidate samples {16, 32, 48}
- **eta**: DTR threshold for deciding whether to extend thinking {0.25, 0.5, 0.75}

Key finding: **eta=0.5** is the optimal threshold, providing the best balance between
extending thinking for hard problems and saving compute on easy ones.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

sns.set_theme(style="whitegrid")

from dtr.aggregation.strategies import (
    SampleResult,
    run_all_strategies,
    run_trials,
    think_at_n,
)
from dtr.aggregation.cost import compute_selective_cost, compute_cost_ratio
from dtr.utils.io import load_json

## Configuration

In [ ]:
MODEL = "deepseek-r1"
BENCHMARKS = ["aime_2025", "hmmt_2025", "gpqa_diamond", "livecodebench"]
BENCHMARK_LABELS = ["AIME 2025", "HMMT 2025", "GPQA Diamond", "LiveCodeBench"]

METRICS_DIR = Path("..") / "results" / "metrics"

N_VALUES = [16, 32, 48]
ETA_VALUES = [0.25, 0.5, 0.75]
N_TRIALS = 50  # Bootstrap trials

## Load Data

In [ ]:
all_data = {}

for bench in BENCHMARKS:
    metrics_path = METRICS_DIR / MODEL / bench / "per_sample_metrics.json"
    if metrics_path.exists():
        data = load_json(str(metrics_path))
        all_data[bench] = data
        print(f"Loaded {bench}: {len(data)} samples")
    else:
        print(f"WARNING: {metrics_path} not found")

print(f"\nBenchmarks loaded: {list(all_data.keys())}")

## Sweep n and eta

For each (n, eta) combination, run Think@n and record accuracy and cost.

In [ ]:
sweep_results = {}  # bench -> {(n, eta): {accuracy, cost, ...}}

for bench in BENCHMARKS:
    if bench not in all_data:
        continue
    
    data = all_data[bench]
    samples = [
        SampleResult(
            answer=d.get("answer", ""),
            correct=d["correct"],
            dtr=d.get("dtr", 0.0),
            token_count=d.get("token_count", 0),
            log_prob=d.get("log_prob", 0.0),
        )
        for d in data
    ]
    
    bench_results = {}
    for n in N_VALUES:
        for eta in ETA_VALUES:
            print(f"  {bench}: n={n}, eta={eta}...", end=" ")
            
            # Run Think@n with specific eta
            trial_results = run_trials(
                samples,
                strategy_fn=lambda s, _n=n, _eta=eta: think_at_n(s, n=_n, eta=_eta),
                n_trials=N_TRIALS,
            )
            
            bench_results[(n, eta)] = {
                "accuracy_mean": np.mean(trial_results["accuracies"]),
                "accuracy_std": np.std(trial_results["accuracies"]),
                "cost_mean": np.mean(trial_results["costs"]),
                "cost_std": np.std(trial_results["costs"]),
            }
            
            acc = bench_results[(n, eta)]["accuracy_mean"]
            cost = bench_results[(n, eta)]["cost_mean"]
            print(f"acc={acc:.1%}, cost={cost:.2f}x")
    
    sweep_results[bench] = bench_results

print("\nSweep complete.")

## Figure 8: Think@n Grid (n x eta)

A grid of subplots showing Think@n accuracy for each (n, eta) combination.
Line plots across benchmarks reveal that eta=0.5 is consistently optimal.

In [ ]:
fig, axes = plt.subplots(
    len(N_VALUES), len(ETA_VALUES),
    figsize=(14, 10),
    sharey=True, sharex=True,
)

fig.suptitle(
    "Figure 8: Think@n Analysis (Accuracy by n and eta)",
    fontsize=16, fontweight="bold", y=1.02,
)

colors = sns.color_palette("Set2", len(BENCHMARKS))

for row, n in enumerate(N_VALUES):
    for col, eta in enumerate(ETA_VALUES):
        ax = axes[row, col]
        
        bench_accs = []
        bench_errs = []
        bench_labels_plot = []
        
        for i, (bench, bench_label) in enumerate(zip(BENCHMARKS, BENCHMARK_LABELS)):
            if bench not in sweep_results:
                continue
            
            if (n, eta) in sweep_results[bench]:
                res = sweep_results[bench][(n, eta)]
                bench_accs.append(res["accuracy_mean"])
                bench_errs.append(res["accuracy_std"])
                bench_labels_plot.append(bench_label)
        
        if bench_accs:
            x_pos = np.arange(len(bench_labels_plot))
            bars = ax.bar(
                x_pos, bench_accs,
                yerr=bench_errs,
                color=colors[:len(bench_accs)],
                edgecolor="white",
                capsize=3,
                width=0.6,
            )
            
            for bar, val in zip(bars, bench_accs):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.01,
                    f"{val:.0%}",
                    ha="center", fontsize=8,
                )
            
            ax.set_xticks(x_pos)
            if row == len(N_VALUES) - 1:
                ax.set_xticklabels(
                    [l.replace(" ", "\n") for l in bench_labels_plot],
                    fontsize=8,
                )
            else:
                ax.set_xticklabels([])
        
        # Title with eta highlight
        title_style = "bold" if eta == 0.5 else "normal"
        title_color = "darkgreen" if eta == 0.5 else "black"
        ax.set_title(
            f"n={n}, eta={eta}",
            fontsize=11, fontweight=title_style, color=title_color,
        )
        
        if col == 0:
            ax.set_ylabel(f"Accuracy (n={n})", fontsize=10)

plt.tight_layout()
plt.savefig("../results/figures/fig8_think_at_n_analysis.pdf", bbox_inches="tight", dpi=300)
plt.show()

## Alternative View: Line Plot Across eta Values

For each benchmark, show how accuracy changes with eta for different n values.

In [ ]:
fig, axes = plt.subplots(1, len(BENCHMARKS), figsize=(16, 4.5), sharey=True)
fig.suptitle(
    "Think@n Accuracy vs eta Threshold",
    fontsize=15, fontweight="bold", y=1.05,
)

n_colors = sns.color_palette("deep", len(N_VALUES))
n_markers = ["o", "s", "D"]

for ax, bench, bench_label in zip(axes, BENCHMARKS, BENCHMARK_LABELS):
    if bench not in sweep_results:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(bench_label, fontsize=12)
        continue
    
    for i, n in enumerate(N_VALUES):
        etas = []
        accs = []
        errs = []
        
        for eta in ETA_VALUES:
            if (n, eta) in sweep_results[bench]:
                res = sweep_results[bench][(n, eta)]
                etas.append(eta)
                accs.append(res["accuracy_mean"])
                errs.append(res["accuracy_std"])
        
        if etas:
            ax.errorbar(
                etas, accs, yerr=errs,
                marker=n_markers[i], color=n_colors[i],
                linewidth=2, markersize=8, capsize=3,
                label=f"n={n}",
            )
    
    # Mark optimal eta
    ax.axvline(0.5, color="gray", linestyle=":", alpha=0.5, label="eta=0.5 (optimal)")
    
    ax.set_title(bench_label, fontsize=12)
    ax.set_xlabel("eta (DTR threshold)", fontsize=11)
    if ax == axes[0]:
        ax.set_ylabel("Accuracy", fontsize=11)
    ax.set_xticks(ETA_VALUES)

handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="lower center", ncol=len(N_VALUES) + 1,
    fontsize=10, bbox_to_anchor=(0.5, -0.1),
)

plt.tight_layout()
plt.show()

## Cost Analysis Across (n, eta)

Show how inference cost varies with eta for different n values.

In [ ]:
fig, axes = plt.subplots(1, len(BENCHMARKS), figsize=(16, 4.5), sharey=True)
fig.suptitle(
    "Think@n Inference Cost vs eta Threshold",
    fontsize=15, fontweight="bold", y=1.05,
)

for ax, bench, bench_label in zip(axes, BENCHMARKS, BENCHMARK_LABELS):
    if bench not in sweep_results:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(bench_label, fontsize=12)
        continue
    
    for i, n in enumerate(N_VALUES):
        etas = []
        costs = []
        
        for eta in ETA_VALUES:
            if (n, eta) in sweep_results[bench]:
                res = sweep_results[bench][(n, eta)]
                etas.append(eta)
                costs.append(res["cost_mean"])
        
        if etas:
            ax.plot(
                etas, costs,
                marker=n_markers[i], color=n_colors[i],
                linewidth=2, markersize=8,
                label=f"n={n}",
            )
    
    ax.axvline(0.5, color="gray", linestyle=":", alpha=0.5)
    ax.set_title(bench_label, fontsize=12)
    ax.set_xlabel("eta (DTR threshold)", fontsize=11)
    if ax == axes[0]:
        ax.set_ylabel("Normalized Cost", fontsize=11)
    ax.set_xticks(ETA_VALUES)

handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="lower center", ncol=len(N_VALUES),
    fontsize=10, bbox_to_anchor=(0.5, -0.1),
)

plt.tight_layout()
plt.show()

## Summary Table: All (n, eta) Combinations

In [ ]:
summary_rows = []
for bench, bench_label in zip(BENCHMARKS, BENCHMARK_LABELS):
    if bench not in sweep_results:
        continue
    for n in N_VALUES:
        for eta in ETA_VALUES:
            if (n, eta) not in sweep_results[bench]:
                continue
            res = sweep_results[bench][(n, eta)]
            summary_rows.append({
                "Benchmark": bench_label,
                "n": n,
                "eta": eta,
                "Accuracy": f"{res['accuracy_mean']:.1%} +/- {res['accuracy_std']:.1%}",
                "Cost": f"{res['cost_mean']:.2f}x +/- {res['cost_std']:.2f}",
            })

summary_df = pd.DataFrame(summary_rows)

# Highlight eta=0.5 rows
def highlight_optimal_eta(row):
    if row["eta"] == 0.5:
        return ["background-color: #c6efce"] * len(row)
    return [""] * len(row)

styled = summary_df.style.apply(highlight_optimal_eta, axis=1)
styled.set_caption("Think@n Performance Across All (n, eta) Combinations")
styled

## Key Takeaway

- **eta=0.5** is the optimal DTR threshold for Think@n across all benchmarks and n values
- Too low (eta=0.25): extends thinking unnecessarily for easy problems, wasting compute
- Too high (eta=0.75): fails to extend thinking for genuinely hard problems, losing accuracy
- Increasing n yields diminishing returns beyond n=32 for most benchmarks